# Stage 1 · Treatment Assignment and Design Validity

Reconstructs PyPI's critical-project designation by re-executing the
production query from Warehouse commit `542f745f2` (paper Sec. 3.2),
validates the reconstruction against an independently built ranking,
removes always-taker projects, tests for manipulation and covariate
balance, and reports a preliminary estimate with a design-validity
summary. Run cells in order; each cell verifies its prerequisites.
All data sources are public; no credentials are required (the optional
BigQuery cross-check needs a Google Cloud project).

---
## Cell 1 — Setup

Installs, parameters, and cache. Safe to re-run at any time.

In [ ]:
# ============================ CELL 1 — SETUP ============================
!pip -q install rdrobust pandas numpy scipy matplotlib requests pyarrow tqdm 2>/dev/null

import os, json, time, math, re, subprocess, urllib.parse, urllib.request
import datetime as dt
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import numpy as np, pandas as pd, requests
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

try:
    from google.colab import drive; drive.mount('/content/drive')
    WORK = Path('/content/drive/MyDrive/msr2027')
except Exception:
    WORK = Path('./msr2027')
CACHE = WORK/'cache'; CACHE.mkdir(parents=True, exist_ok=True)

# ---- Study parameters: the single source of truth ----------------------
ANNOUNCE_DATE    = dt.date(2022, 7, 8)   # PyPI critical-project announcement
COHORT_SIZE      = 3800                  # TWOFACTORMANDATE_COHORTSIZE (~top 1%)
COHORT_GRID      = [3500, 3600, 3700, 3800, 3900, 4000]
WAREHOUSE_COMMIT = '542f745f2ce03e01332516e9a20c2f2ae6b35742'
CONTACT_EMAIL    = 'anonymous.artifact@example.com'   # sent in User-Agent, per API etiquette

# Reproduce PyPI's window: DATE_TRUNC(DATE_SUB(d, INTERVAL 6 MONTH), MONTH) .. d
_y, _m = ANNOUNCE_DATE.year, ANNOUNCE_DATE.month - 6
while _m <= 0: _m += 12; _y -= 1
WIN_START, WIN_END = dt.date(_y,_m,1), ANNOUNCE_DATE
WIN_MONTHS = pd.date_range(WIN_START, WIN_END, freq='MS').strftime('%Y-%m-01').tolist()

SETUP_DONE = True
print('CELL 1 OK')
print(f'  cache        : {CACHE}')
print(f'  PyPI window  : {WIN_START} .. {WIN_END}  ({(WIN_END-WIN_START).days} days)')
print(f'  months       : {WIN_MONTHS}')
print(f'  cohort size  : {COHORT_SIZE}')

---
## Cell 2 — Get the download data

This cell selects between the free and the paid data route.

It tries **ClickHouse** first: a public PyPI download mirror, `demo` user, no account, no quota. If that works the study costs nothing.

If it fails it falls back to **BigQuery**, which is what PyPI itself used. It always does a **dry run first** and asks before firing — a real snapshot from July 2022 records `bytes_billed: 430,686,863,360`, i.e. **431 GB for one 30-day query**, so your 7-month window is ~2.6–3.0 TB (~$15). Never fire that blind.

The query is PyPI's production SQL, with the window pinned to our date instead of `CURRENT_DATE()`. **Do not "clean" the counts** — PyPI used a raw `COUNT(*)` with no installer or mirror filtering, and a cleaner measure would misclassify projects at the cutoff and destroy the design.

In [ ]:
# ======================= CELL 2 — DOWNLOAD DATA ========================
if not globals().get('SETUP_DONE'): raise SystemExit('>>> Run CELL 1 first.')

RANK_PATH = CACHE/f'ranking_{WIN_START}_{WIN_END}.parquet'
CH = 'https://sql-clickhouse.clickhouse.com'

def ch(sql, fmt='TabSeparated', timeout=600):
    p = urllib.parse.urlencode({'user':'demo','default_format':fmt})
    req = urllib.request.Request(CH+'?'+p, data=sql.encode(), method='POST')
    return urllib.request.urlopen(req, timeout=timeout).read().decode()

rank, SOURCE = None, None

if RANK_PATH.exists():
    rank, SOURCE = pd.read_parquet(RANK_PATH), 'cache'
    print(f'Loaded cached ranking: {len(rank):,} projects')
else:
    # ---------- Route A: ClickHouse (free) ----------
    try:
        print('Probing ClickHouse...')
        print(ch('SHOW TABLES FROM pypi'))
        print(ch('DESCRIBE TABLE pypi.pypi_downloads_per_month'))
        sql = f"""SELECT project, SUM(count) AS num_downloads
                  FROM pypi.pypi_downloads_per_month
                  WHERE month IN ({','.join(repr(m) for m in WIN_MONTHS)})
                  GROUP BY project ORDER BY num_downloads DESC LIMIT 200000"""
        rank = pd.DataFrame(json.loads(ch(sql,'JSON'))['data'])
        rank['num_downloads'] = rank['num_downloads'].astype('int64')
        SOURCE = 'clickhouse'
        print(f'\n>>> ClickHouse WORKED. {len(rank):,} projects. Cost: $0.')
    except Exception as e:
        print('ClickHouse unavailable ->', repr(e)[:300])
        print('>>> Falling back to BigQuery.\n')

    # ---------- Route B: BigQuery (authoritative, ~$15) ----------
    if rank is None:
        from google.colab import auth; auth.authenticate_user()
        from google.cloud import bigquery
        client = bigquery.Client(project=input('GCP project id: ').strip())
        sql = f"""SELECT COUNT(*) AS num_downloads, file.project AS project
                  FROM `bigquery-public-data.pypi.file_downloads`
                  WHERE DATE(timestamp) BETWEEN DATE('{WIN_START}') AND DATE('{WIN_END}')
                  GROUP BY file.project ORDER BY num_downloads DESC"""
        est = client.query(sql, job_config=bigquery.QueryJobConfig(dry_run=True, use_query_cache=False))
        tb = est.total_bytes_processed/1e12
        print(f'DRY RUN: {tb:.2f} TB  ->  approx ${tb*6.25:.2f}')
        if input('Proceed? type YES: ').strip() != 'YES':
            raise SystemExit('Aborted by user.')
        rank, SOURCE = client.query(sql).to_dataframe(), 'bigquery'

    rank = rank.sort_values('num_downloads', ascending=False).reset_index(drop=True)
    rank['rank'] = np.arange(1, len(rank)+1)
    rank.to_parquet(RANK_PATH)

CUT_DL = int(rank.loc[rank['rank']==COHORT_SIZE,'num_downloads'].iloc[0])
DATA_DONE = True
print(f'\nCELL 2 OK  (source={SOURCE})')
print(f'  projects ranked : {len(rank):,}')
print(f'  rank 1          : {rank.iloc[0]["project"]}  ({rank.iloc[0]["num_downloads"]:,})')
print(f'  CUTOFF rank {COHORT_SIZE} : {rank.iloc[COHORT_SIZE-1]["project"]}  ({CUT_DL:,} downloads)')

---
## Cell 3 — Validate the reconstruction

Before trusting any estimate, prove the reconstruction is right. `hugovk/top-pypi-packages` publishes monthly top-N snapshots built independently from BigQuery, and the 1 July 2022 snapshot sits exactly at our announcement date.

It uses a **30-day** window where PyPI used **7 months**, so the rankings will not match exactly — and should not. We are testing whether they agree *well enough near the cutoff* to be credible. This is also your answer to the reviewer who asks how you know your reconstruction reproduces PyPI's designation.

In [ ]:
# ======================= CELL 3 — VALIDATION ===========================
if not globals().get('DATA_DONE'): raise SystemExit('>>> Run CELL 2 first.')
from scipy.stats import spearmanr

if not Path('/content/tpp').exists():
    subprocess.run(['git','clone','-q','--filter=blob:none',
                    'https://github.com/hugovk/top-pypi-packages.git','/content/tpp'])
snap = subprocess.run(['git','-C','/content/tpp','show',
                       'dd326bf:top-pypi-packages-30-days.min.json'],
                      capture_output=True, text=True).stdout
tpp = pd.DataFrame(json.loads(snap)['rows'])
tpp['tpp_rank'] = np.arange(1, len(tpp)+1)

m = rank.merge(tpp[['project','tpp_rank']], on='project', how='inner')
RHO = spearmanr(m['rank'], m['tpp_rank']).correlation
near = m[(m['rank']>COHORT_SIZE*0.5)&(m['rank']<COHORT_SIZE*1.5)]
RHO_NEAR = spearmanr(near['rank'], near['tpp_rank']).correlation
AGREE = ((m['rank']<=COHORT_SIZE)==(m['tpp_rank']<=COHORT_SIZE)).mean()

print(f'Independent snapshot 2022-07-01: {len(tpp):,} projects; overlap n={len(m):,}')
print(f'  Spearman rho, overall        : {RHO:.4f}')
print(f'  Spearman rho, near cutoff    : {RHO_NEAR:.4f}  (n={len(near):,})')
print(f'  Treatment-status agreement   : {AGREE:.1%}')
VALID_DONE = True

---
## Cell 4 — Remove the always-takers

**The cell that saves the study.** PyPI's task does:

```python
project_names = our_dependencies | top_projects
```

Warehouse's own production dependencies were designated critical **regardless of rank**. A project ranked 40,000th could be treated because PyPI depends on it. Left in, these sit on the control side of the cutoff while actually being treated — contamination that could manufacture or destroy the entire result.

In [ ]:
# ===================== CELL 4 — ALWAYS-TAKERS ==========================
if not globals().get('VALID_DONE'): raise SystemExit('>>> Run CELL 3 first.')

if not Path('/content/wh').exists():
    subprocess.run(['git','clone','-q','--filter=blob:none',
                    'https://github.com/pypi/warehouse.git','/content/wh'])
subprocess.run(['git','-C','/content/wh','checkout','-q',WAREHOUSE_COMMIT])

always = set()
for f in ['requirements/main.txt','requirements/deploy.txt']:
    p = Path('/content/wh')/f
    if not p.exists(): print('MISSING:', f); continue
    for line in p.read_text().splitlines():
        line = line.split('#')[0].strip()
        if not line or line.startswith('-'): continue
        n = re.split(r'[=<>!\[;~]', line)[0].strip()
        if n: always.add(n.lower().replace('_','-'))

rank['norm'] = rank['project'].str.lower().str.replace('_','-', regex=False)
rank['always_taker'] = rank['norm'].isin(always)
contam = rank[(rank['rank']>COHORT_SIZE) & rank['always_taker']]

print(f'Warehouse production dependencies : {len(always)}')
print(f'Of these, ranked BELOW the cutoff : {len(contam)}   <-- would contaminate controls')
if len(contam): print(contam[['project','rank','num_downloads']].head(20).to_string(index=False))

rank_clean = rank[~((rank['rank']>COHORT_SIZE) & rank['always_taker'])].copy()
CLEAN_DONE = True
print(f'\nCELL 4 OK — analysis frame: {len(rank_clean):,} projects '
      f'({len(rank)-len(rank_clean)} removed)')

---
## Cell 5 — Analysis band and manipulation test

A subtlety worth getting right: the assignment rule is a **rank** limit, and the density of rank is uniform by construction, so a McCrary test on rank is vacuous. The test must run on **downloads**, the underlying continuous variable.

But there is a second trap, and I fell into it on the first draft of this notebook. Testing whether counts just-above and just-below the cutoff are *equal* assumes a flat density. PyPI downloads follow a steep power law, so **more projects sit just below any threshold you pick** — the naive test returns p = 0.001 at the true cutoff and would falsely signal manipulation here.

The fix is placebo calibration: compute the below/above ratio at the real cutoff, then compute the same statistic at dozens of fake cutoffs elsewhere in the distribution, and ask how unusual the real one is. On the July 2022 snapshot this moves the verdict from a spurious p = 0.001 to a correctly-specified **p = 0.12** — no evidence of manipulation.

Is manipulation even plausible? Barely: maintainers did not know the cutoff, and download counts are driven by other people's CI. But PyPI's query does no installer filtering, so inflation was *possible*. Test it rather than assert it.

In [ ]:
# ================== CELL 5 — BAND + MANIPULATION =======================
if not globals().get('CLEAN_DONE'): raise SystemExit('>>> Run CELL 4 first.')

band = rank_clean[(rank_clean['num_downloads']>CUT_DL*0.5) &
                  (rank_clean['num_downloads']<CUT_DL*2.0)].copy()
band['treated'] = (band['rank']<=COHORT_SIZE).astype(int)
band['running'] = np.log(band['num_downloads']) - np.log(CUT_DL)   # 0 at cutoff

# ---- Placebo-calibrated density test ----------------------------------
# NOT a binomial test against 0.5: the download distribution is a steep power
# law, so more projects sit just below ANY threshold. Calibrate against fake
# cutoffs instead of against a flat density.
H = 0.15
def _ratio(c):
    run = np.log(rank_clean['num_downloads']) - np.log(c)
    lo = int(((run>-H)&(run<0)).sum()); hi = int(((run>=0)&(run<H)).sum())
    return lo, hi, (hi/lo if lo else np.nan)

LO, HI, RATIO = _ratio(CUT_DL)
placebo = []
for pr in range(int(COHORT_SIZE*0.3), int(COHORT_SIZE*2.5), 50):
    if abs(pr-COHORT_SIZE) < 500: continue          # exclude thetrue cutoff's neighbourhood
    sel = rank_clean.loc[rank_clean['rank']==pr, 'num_downloads']
    if not len(sel): continue
    l, h2, r2 = _ratio(float(sel.iloc[0]))
    if np.isfinite(r2) and l > 20: placebo.append(r2)
placebo = np.array(placebo)

if len(placebo) >= 20:
    med = float(np.median(placebo))
    P_DENSITY = float((np.abs(placebo-med) >= abs(RATIO-med)).mean())
else:
    med, P_DENSITY = float('nan'), float('nan')
    print('WARNING: too few placebo cutoffs; density test inconclusive.')

fig, ax = plt.subplots(1, 2, figsize=(13,3.8))
ax[0].hist(band['running'], bins=80, color='#4C78A8'); ax[0].axvline(0, color='crimson', lw=2)
ax[0].set_title('Running variable density at cutoff'); ax[0].set_xlabel('log(downloads) − log(cutoff)')
if len(placebo)>=20:
    ax[1].hist(placebo, bins=30, color='#B0B0B0'); ax[1].axvline(RATIO, color='crimson', lw=2)
    ax[1].set_title('Above/below ratio: true cutoff vs placebos'); ax[1].set_xlabel('ratio')
plt.tight_layout(); plt.show()

print(f'Band [0.5c, 2c]: {len(band):,} projects '
      f'({band.treated.sum():,} treated / {(1-band.treated).sum():,} control)')
print(f'Density at true cutoff : {LO} below / {HI} above  -> ratio {RATIO:.4f}')
print(f'Placebo cutoffs (n={len(placebo)}) : median ratio {med:.4f}')
print(f'Empirical p-value      : {P_DENSITY:.3f}')
print('  ' + ('CONCERN: possible manipulation.' if P_DENSITY<0.05 else 'No evidence of manipulation.'))
BAND_DONE = True; print('\nCELL 5 OK')

---
## Cell 6 — Outcomes

All ground-truth recorded events, no inferred constructs. Two sources, because PyPI has been deprecating the `releases` key in its JSON API: the code tries that first and falls back to the PEP 691/700 simple JSON index, which carries `upload-time` per file.

**A limitation to state plainly:** PyPI exposes no public API for project collaborators, so **maintainer-set churn is not observable**. I had it in the earlier outcome list and it has to come out. Substituting the linked GitHub repo's contributor set would be an inferred proxy, which is exactly the construct problem you ruled out. The outcomes here are therefore release cadence and abandonment; fuller outcome margins are collected in notebook 02.

In [ ]:
# ======================= CELL 6 — OUTCOMES =============================
if not globals().get('BAND_DONE'): raise SystemExit('>>> Run CELL 5 first.')

N_SAMPLE, POST_DAYS = 1500, 365          # raise to len(band) for the full run
OUT_PATH = CACHE/f'outcomes_{N_SAMPLE}.parquet'
T0 = pd.Timestamp(ANNOUNCE_DATE)

sess = requests.Session()
sess.headers.update({'User-Agent': f'MSR2027-research ({CONTACT_EMAIL})'})

def upload_times(project):
    try:
        r = sess.get(f'https://pypi.org/pypi/{project}/json', timeout=20)
        if r.status_code == 200:
            rel = r.json().get('releases')
            if rel:
                ts = [pd.Timestamp(f['upload_time_iso_8601']).tz_localize(None)
                      for v in rel.values() for f in v[:1] if f.get('upload_time_iso_8601')]
                if ts: return sorted(ts)
    except Exception: pass
    try:  # PEP 691/700 fallback
        r = sess.get(f'https://pypi.org/simple/{project}/', timeout=20,
                     headers={'Accept':'application/vnd.pypi.simple.v1+json'})
        if r.status_code == 200:
            ts = [pd.Timestamp(f['upload-time']).tz_localize(None)
                  for f in r.json().get('files',[]) if f.get('upload-time')]
            if ts: return sorted(ts)
    except Exception: pass
    return None

def rec(project):
    ts = upload_times(project)
    if not ts: return None
    s = pd.Series(ts)
    post = s[(s>=T0)&(s<T0+pd.Timedelta(days=POST_DAYS))]
    return dict(project=project, age_days=(T0-s.iloc[0]).days,
        n_rel_pre = int(((s>=T0-pd.Timedelta(days=POST_DAYS))&(s<T0)).sum()),
        n_rel_post= int(len(post)),
        days_to_next = (post.iloc[0]-T0).days if len(post) else POST_DAYS,
        abandoned = int(s.iloc[-1] < T0 + pd.Timedelta(days=730)))

if OUT_PATH.exists():
    df = pd.read_parquet(OUT_PATH); print(f'Loaded cached outcomes: {len(df):,}')
else:
    sample = band.sample(min(N_SAMPLE,len(band)), random_state=20261019)
    with ThreadPoolExecutor(max_workers=12) as ex:
        recs = list(tqdm(ex.map(rec, sample['project']), total=len(sample)))
    got = pd.DataFrame([r for r in recs if r])
    df = sample.merge(got, on='project'); df.to_parquet(OUT_PATH)
    HIT = len(df)/len(sample)
    print(f'Collected {len(df):,} / {len(sample):,}  (hit rate {HIT:.0%})')

HIT = len(df)/min(N_SAMPLE,len(band))
OUTCOMES_DONE = True; print('\nCELL 6 OK')

---
## Cell 7 — Balance and the estimate

Covariates are pre-determined: project age and pre-period release count **must not jump** at the cutoff. If they do, the cutoff is proxying for something else and the design is dead.

Then the primary estimate, with a sensitivity sweep over cohort size — the deployed `TWOFACTORMANDATE_COHORTSIZE` is environment config and not in the repository, so it is tested, not assumed.

In [ ]:
# ================== CELL 7 — BALANCE + ESTIMATE ========================
if not globals().get('OUTCOMES_DONE'): raise SystemExit('>>> Run CELL 6 first.')
from rdrobust import rdrobust, rdplot

def rd(y, x):
    r = rdrobust(y=np.asarray(y,float), x=np.asarray(x,float), c=0)
    try:    n_eff = int(np.sum(np.asarray(r.N_h)))
    except Exception: n_eff = len(y)
    return float(r.coef.iloc[0,0]), float(r.pv.iloc[2,0]), n_eff

print('='*66); print('COVARIATE BALANCE — these should be NULL'); print('='*66)
BALANCE_OK = True
for cov in ['age_days','n_rel_pre']:
    d = df.dropna(subset=[cov])
    est,pv,_ = rd(d[cov], d['running'])
    ok = pv >= 0.05; BALANCE_OK &= ok
    print(f'  {cov:12s} jump={est:10.3f}   robust p={pv:.3f}   [{"ok" if ok else "PROBLEM"}]')

print('\n'+'='*66); print('PRIMARY: releases in 12 months after designation'); print('='*66)
d = df.dropna(subset=['n_rel_post'])
EST, P_EST, N_EFF = rd(d['n_rel_post'], d['running'])
print(f'  estimate = {EST:.4f} releases   robust p = {P_EST:.4f}   effective n = {N_EFF}')

try:
    rdplot(y=d['n_rel_post'].values.astype(float), x=d['running'].values.astype(float), c=0,
           title='Releases, 12 months post-designation',
           x_label='log(downloads) − log(cutoff)', y_label='releases'); plt.show()
except Exception as e: print('  (rdplot skipped:', e, ')')

print('\n'+'='*66); print('SENSITIVITY TO COHORT SIZE'); print('='*66)
for cs in COHORT_GRID:
    try:
        c2 = float(rank_clean.loc[rank_clean['rank']==cs,'num_downloads'].iloc[0])
        dd = df.assign(r2=np.log(df['num_downloads'])-np.log(c2))
        dd = dd[(dd.r2>-0.7)&(dd.r2<0.7)]
        e2,p2,n2 = rd(dd['n_rel_post'], dd['r2'])
        print(f'  cohort={cs}: est={e2:9.4f}  p={p2:.3f}  n={n2}')
    except Exception as e: print(f'  cohort={cs}: failed ({e})')
EST_DONE = True; print('\nCELL 7 OK')

---
## Cell 8 — design-validity summary

Read this honestly. A **null is a perfectly good paper** — "designation did not measurably reduce maintainer activity" is a result PyPI, npm and the CRA drafters would all cite. What is not survivable is a design that cannot estimate anything: covariates that jump, evidence of manipulation, or a bandwidth so narrow nothing is identified.

The decision rule is fixed here *before* you read the estimate.

In [ ]:
# ======================== CELL 8 — design-validity summary ==========================
if not globals().get('EST_DONE'): raise SystemExit('>>> Run CELL 7 first.')

checks = {
 'Ranking validated vs independent source (rho>0.90)': RHO > 0.90,
 'Always-takers identified and removed'              : True,
 'No evidence of manipulation (density p>0.05)'      : P_DENSITY > 0.05,
 'Band adequately populated (n>800)'                 : len(band) > 800,
 'Outcome coverage acceptable (>80%)'                : HIT > 0.80,
 'Covariates balanced at cutoff'                     : BALANCE_OK,
 'RDD estimable (effective n>200)'                   : N_EFF > 200,
}
print('='*66); print('  DESIGN-VALIDITY SUMMARY'); print('='*66)
for k,v in checks.items(): print(f'  [{"PASS" if v else "FAIL"}]  {k}')

print('\n'+'-'*66)
if all(checks.values()):
    print('  >>> All design checks passed. Proceed to notebook 02.')
    print(f'      Preliminary: {EST:.4f} releases, robust p={P_EST:.4f}, n_eff={N_EFF}')
    print('      A null here is a publishable finding, not a failure.')
else:
    print('  >>> One or more design checks failed; the RDD is not valid under this specification.')
    print('      no BigQuery, and the data is a single git clone.')
print('-'*66)